In [1]:
import pandas as pd
import altair as alt
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [10]:
df = pd.read_csv("https://uiuc-ischool-dataviz.github.io/spring2019online/week04/data/ufo-scrubbed-geocoded-time-standardized-00.csv",
                  names = ["date", "city", "state", "country", "shape", "duration_seconds", "duration", "comment", "report_date", "latitude", "longitude"],
                  parse_dates = ["date", "report_date"])

In [11]:
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce").round(3)
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce").round(3)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["latitude","longitude","date"])
df["year"] = df["date"].dt.year

In [5]:
df.head()

,date,city,state,country,shape,duration_seconds,duration,comment,report_date,latitude,longitude,year
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,2004-04-27,29.883,-97.941,1949
1,1949-10-10 21:00:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,2005-12-16,29.384,-98.581,1949
2,1955-10-10 17:00:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,2008-01-21,53.200,-2.917,1955
3,1956-10-10 21:00:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,2004-01-17,28.978,-96.646,1956
4,1960-10-10 20:00:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,2004-01-22,21.418,-157.804,1960


In [12]:
us = df[(df["country"]=="us") & df["state"].notna()].copy()
us["state"] = us["state"].str.upper()

top_shapes = us["shape"].value_counts().head(6).index
us["shape_group"] = us["shape"].where(us["shape"].isin(top_shapes), "other").fillna("other")

us = us[["city","state","year","shape_group","latitude","longitude"]]

state_counts = (us.groupby("state").size().reset_index(name="sightings").nlargest(15, "sightings"))
len(us), state_counts.head()

(65114,
    state  sightings
 4     CA       8912
 48    WA       3966
 9     FL       3835
 44    TX       3447
 34    NY       2980)

In [13]:
states = alt.topo_feature("https://cdn.jsdelivr.net/npm/vega-datasets@v1.29.0/data/us-10m.json", "states")

# interactions
shape_sel = alt.selection_point(fields=["shape_group"], bind="legend")   # click legend
state_sel = alt.selection_point(fields=["state"], name="pick_state")      # click bar
year_slider = alt.param(name="min_year", value=1990,
                        bind=alt.binding_range(min=1940, max=2014, step=1,
                                               name="Sightings since year: "))

# Plot 1: point map
base = (alt.Chart(states).mark_geoshape(fill="#eeeeee", stroke="white")
        .project("albersUsa").properties(width=650, height=400))

points = (alt.Chart(us).mark_circle(size=12).encode(
        longitude="longitude:Q", latitude="latitude:Q",
        color=alt.Color("shape_group:N", scale=alt.Scale(scheme="tableau10"),
                        title="Shape (click to filter)"),
        opacity=alt.condition(shape_sel, alt.value(0.6), alt.value(0.03)),
        tooltip=["city:N","state:N","year:Q","shape_group:N"])
    .add_params(shape_sel, year_slider)
    .transform_filter(alt.datum.year >= year_slider)
    .transform_filter(state_sel))

map_chart = (base + points).properties(title="Where UFOs are reported across the US")

# Plot 2: bar chart of top states, click a bar to filter the map
bars = (alt.Chart(state_counts).mark_bar().encode(
        x=alt.X("sightings:Q", title="Total sightings"),
        y=alt.Y("state:N", sort="-x", title="State"),
        color=alt.condition(state_sel, alt.value("#d95f02"), alt.value("#7a7a7a")),
        tooltip=["state","sightings"])
    .add_params(state_sel)
    .properties(width=250, height=400, title="Top 15 states (click to filter map)"))

chart = map_chart | bars
chart

alt.HConcatChart(...)